In [ ]:
# notebooks/03_pass_map.ipynb  セル1
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from mplsoccer import Pitch

# src フォルダを import の探索先に追加する(自作の loader.py などを使うため)
sys.path.append("../src")

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

# 図の色(暗い緑の背景に合う色。青とオレンジは、色の見分けにくい人にも区別しやすい組み合わせ)
BG_COLOR = "#22312b"      # 背景・ピッチの色
LINE_COLOR = "#c7d5cc"    # ピッチのラインの色
TEXT_COLOR = "#e8efe9"    # 文字の色
OK_COLOR = "#6cb2eb"      # 成功パスの色(青)
NG_COLOR = "#f28e4b"      # 失敗パスの色(オレンジ)


In [ ]:
# notebooks/03_pass_map.ipynb  セル2
from pitchviz.loader import get_events

TEAM_NAME = "Japan"       # フェーズ2のメモの値に置き換える(例)
MATCH_ID = 7584           # フェーズ2のメモの値に置き換える(例)
MATCH_DATE = "2018-07-02" # フェーズ2のメモの日付に置き換える(例)

events = get_events(MATCH_ID)
teams = list(events["team"].unique())
opponent_name = [t for t in teams if t != TEAM_NAME][0]

print(events.shape)
print(teams, "->", TEAM_NAME, "vs", opponent_name)


In [ ]:
# notebooks/03_pass_map.ipynb  セル3
passes = events[(events["type"] == "Pass") & (events["team"] == TEAM_NAME)].copy()

passes["x"] = passes["location"].str[0]
passes["y"] = passes["location"].str[1]
passes["end_x"] = passes["pass_end_location"].str[0]
passes["end_y"] = passes["pass_end_location"].str[1]
passes["is_complete"] = passes["pass_outcome"].isna()

print("passes:", len(passes), " complete:", int(passes["is_complete"].sum()))
print("mean forward (end_x - x):", round((passes["end_x"] - passes["x"]).mean(), 1))
passes[["minute", "player", "x", "y", "end_x", "end_y", "pass_outcome", "is_complete"]].head()


In [ ]:
# notebooks/03_pass_map.ipynb  セル4
pitch = Pitch(pitch_type="statsbomb", pitch_color=BG_COLOR, line_color=LINE_COLOR)
fig, ax = pitch.draw(figsize=(12, 8))
fig.set_facecolor(BG_COLOR)

first = passes.head(1)   # 先頭の1行(1本のパス)
pitch.arrows(first.x, first.y, first.end_x, first.end_y,
             width=2, headwidth=8, headlength=8, color=OK_COLOR, ax=ax)
plt.show()


In [ ]:
# notebooks/03_pass_map.ipynb  セル5
pitch = Pitch(pitch_type="statsbomb", pitch_color=BG_COLOR, line_color=LINE_COLOR)
fig, ax = pitch.draw(figsize=(12, 8))
fig.set_facecolor(BG_COLOR)

pitch.arrows(passes.x, passes.y, passes.end_x, passes.end_y,
             width=2, headwidth=6, headlength=6, color=OK_COLOR, alpha=0.8, ax=ax)
plt.show()


In [ ]:
# notebooks/03_pass_map.ipynb  セル6(課題1)
pitch = Pitch(pitch_type="statsbomb", pitch_color=BG_COLOR, line_color=LINE_COLOR)
fig, ax = pitch.draw(figsize=(12, 8))
fig.set_facecolor(BG_COLOR)

complete = passes[passes["is_complete"]]      # 成功したパスだけ
failed = passes[~passes["is_complete"]]       # 失敗したパスだけ

pitch.arrows(complete.x, complete.y, complete.end_x, complete.end_y,
             width=2, headwidth=6, headlength=6, color=OK_COLOR, alpha=0.85, ax=ax)
pitch.arrows(failed.x, failed.y, failed.end_x, failed.end_y,
             width=2, headwidth=6, headlength=6, color=NG_COLOR, alpha=0.85, ax=ax)
plt.show()


In [ ]:
# notebooks/03_pass_map.ipynb  セル7(課題2)
n_total = len(passes)       # 全部のパスの数
n_complete = len(complete)    # 成功したパスの数
rate = n_complete / n_total * 100

print(f"{TEAM_NAME}: {n_complete}/{n_total} passes completed ({rate:.1f}%)")
print(len(complete) + len(failed) == len(passes))


In [ ]:
# notebooks/03_pass_map.ipynb  セル8
pitch = Pitch(pitch_type="statsbomb", pitch_color=BG_COLOR, line_color=LINE_COLOR)
fig, axs = pitch.grid(figheight=9, title_height=0.08, grid_height=0.80, endnote_height=0.07,
                      title_space=0, endnote_space=0, axis=False)
fig.set_facecolor(BG_COLOR)
ax = axs["pitch"]

complete = passes[passes["is_complete"]]
failed = passes[~passes["is_complete"]]

pitch.arrows(complete.x, complete.y, complete.end_x, complete.end_y,
             width=2, headwidth=6, headlength=6, color=OK_COLOR, alpha=0.85,
             label=f"Completed ({len(complete)})", ax=ax)
pitch.arrows(failed.x, failed.y, failed.end_x, failed.end_y,
             width=2, headwidth=6, headlength=6, color=NG_COLOR, alpha=0.85,
             label=f"Failed ({len(failed)})", ax=ax)

ax.legend(facecolor=BG_COLOR, edgecolor=LINE_COLOR, labelcolor=TEXT_COLOR,
          loc="upper left", fontsize=12)

axs["title"].text(0.5, 0.68, f"{TEAM_NAME} pass map vs {opponent_name}",
                  color=TEXT_COLOR, ha="center", va="center", fontsize=24,
                  transform=axs["title"].transAxes)
axs["title"].text(0.5, 0.18, f"{MATCH_DATE}   |   attacking left to right",
                  color=LINE_COLOR, ha="center", va="center", fontsize=13,
                  transform=axs["title"].transAxes)
axs["endnote"].text(0.01, 0.5, "Data: StatsBomb Open Data",
                    color=TEXT_COLOR, ha="left", va="center", fontsize=11,
                    transform=axs["endnote"].transAxes)
plt.show()


In [ ]:
# notebooks/03_pass_map.ipynb  セル9
OUTPUT_DIR = Path("../outputs")
out_path = OUTPUT_DIR / "pass_map_japan_belgium.png"   # 例。自分の試合に合わせて変える

fig.savefig(out_path, dpi=200, facecolor=fig.get_facecolor())
print("saved:", out_path)


In [ ]:
# notebooks/03_pass_map.ipynb  セル10(課題3)
def add_logo(fig, anchor_ax, logo_path, height_ratio=0.8, margin=0.01):
    """anchor_ax(ロゴを置く欄)の右端に、ロゴ画像を縦横比そのままで置く。"""
    logo_path = Path(logo_path)
    if not logo_path.exists():
        print(f"[warn] logo not found: {logo_path}")
        return None

    img = plt.imread(logo_path)               # 画像ファイルを読み込む
    img_h, img_w = img.shape[:2]             # 画像の(縦, 横)のピクセル数

    box = anchor_ax.get_position()              # 欄の位置(図全体を0〜1とした割合)
    fig_w, fig_h = fig.get_size_inches()    # 図の大きさ(インチ)

    logo_h = box.height * height_ratio                    # ロゴの高さ(図に対する割合)
    logo_w = logo_h * (img_w / img_h) * (fig_h / fig_w)   # 縦横比を保った幅(割合)
    left = box.x1 - logo_w - margin                       # 欄の右端から、少し内側
    bottom = box.y0 + (box.height - logo_h) / 2           # 欄の上下の中央

    logo_ax = fig.add_axes([left, bottom, logo_w, logo_h])    # ロゴ用の描く場所を追加する
    logo_ax.imshow(img)
    logo_ax.axis("off")
    return logo_ax


LOGO_PATH = "../assets/statsbomb_logo_white.png"
add_logo(fig, axs["endnote"], LOGO_PATH)
fig


In [ ]:
# notebooks/03_pass_map.ipynb  セル11
from pitchviz.loader import get_passes
from pitchviz.pass_map import draw_pass_map, make_filename, save_figure

print(make_filename("Japan", "Belgium"))
print(make_filename("Japan Women's", "Norway Women's"))
print(len(get_passes(events, TEAM_NAME)) == len(passes))


In [ ]:
# notebooks/03_pass_map.ipynb  セル12
LOGO_PATH = "../assets/statsbomb_logo_white.png"
OUTPUT_DIR = Path("../outputs")

for team in teams:
    team_passes = get_passes(events, team)
    opponent = [t for t in teams if t != team][0]

    fig = draw_pass_map(team_passes, team, opponent, MATCH_DATE, logo_path=LOGO_PATH)
    out_path = save_figure(fig, OUTPUT_DIR / make_filename(team, opponent))
    print("saved:", out_path)
